In [ ]:
## this notebook is a bit of a mess .. was not necessarily rerun end-to-end 

import os
import warnings
import copy
import numpy as np
import pandas as pd
import sqlite3
import healpy as hp
import matplotlib.pyplot as plt
import skyproj
from IPython.display import display, HTML
from tabulate import tabulate
import datetime

from astropy.time import Time, TimeDelta
import astropy.units as u
import datetime

from rubin_scheduler.scheduler import sim_runner
from rubin_scheduler.scheduler.model_observatory import ModelObservatory
from rubin_scheduler.scheduler.schedulers import SimpleBandSched, CoreScheduler
from rubin_scheduler.scheduler.features import Conditions
from rubin_scheduler.scheduler.utils import SchemaConverter, run_info_table

import rubin_sim.maf as maf
from rubin_sim.data import get_baseline

from rubin_nights import connections
import rubin_nights.dayobs_utils as rn_dayobs
import rubin_nights.plot_utils as rn_plots
import rubin_nights.augment_visits as augment_visits
import rubin_nights.rubin_scheduler_addons as rn_sch
import rubin_nights.rubin_sim_addons as rn_sim
from rubin_nights.targets_and_visits import targets_and_visits



import importlib

from sv_survey import sv_support as svs

import lsst.ts.fbs.utils.maintel.sv_config as svc
_ = importlib.reload(svs)
_ = importlib.reload(svc)


band_colors = rn_plots.PlotStyles.band_colors


In [ ]:
# just for plotting purposes later (background)
from rubin_scheduler.scheduler.utils import get_current_footprint
nside = 64
fp, labels = get_current_footprint(nside=nside)
tsurvey_info = svs.survey_times(verbose=True, no_downtime=True)
tsurvey_info.update(svc.survey_footprint(survey_start_mjd=tsurvey_info["survey_start"].mjd, nside=nside))

pp = tsurvey_info["skymap"]["map"]
alpha = np.where(pp >= 0, 1, fp['r'])
alpha = np.where(alpha > 1, 1, alpha)
bg_fp = np.where(fp['r'] == 0, np.nan, fp['r'])
bg_fp = np.where(bg_fp > 1, 1, bg_fp)
sv_fp = np.where(tsurvey_info['fp_array']['r'] > 0, 1, np.nan)
sv_fp = np.where(tsurvey_info['extra_templates_array']['i'] > 0, 0.5, sv_fp)

def make_sv_plot(metric_bundle, proj='laea', vmin=None, vmax=None, ax=None, add_bg=True, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))
    else:
        fig = ax.get_figure()
    
    if proj == 'laea':
        sp = skyproj.LaeaSkyproj( 
            ax=ax,
            celestial=True,
            galactic=False,
            gridlines=True,
            n_grid_lon=8,
            n_grid_lat=5,
            lat_0=-90,
            lon_0=0,
            extent=[0.0, 360.0, -90, 85],
        )
        # Laea only shows half the sky if zoom = True
        # due to bug in current skyproj
        zoom = False
    else: 
        sp = skyproj.McBrydeSkyproj( 
            ax=ax,
            celestial=True,
            galactic=False,
            gridlines=True,
            n_grid_lon=8,
            n_grid_lat=7,
            lon_0=0,
        )
        zoom = True

    if fig is not None and proj == 'laea':
        sp.ax.set_xlabel("R.A.", fontsize=12, labelpad=9)
        sp.ax.set_ylabel("Dec.", fontsize=12, labelpad=12)

    if add_bg:
        mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
            bg_fp, cmap='Greys', vmin=-1, vmax=4, nest=False, zoom=False, zorder=0
        )
        
        mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
            sv_fp, cmap='Blues', vmin=0, vmax=3, nest=False, zoom=False, zorder=1
        )
        zoom = False
    
    mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
        metric_bundle.metric_values.filled(np.nan), vmin=vmin, vmax=vmax, nest=False, zoom=zoom, zorder=1.5
    )

    if vmin is None and vmax is None:
        extend = None
    elif vmin is None and vmax is not None:
        extend = "max"
    elif vmin is not None and vmax is None:
        extend = "min"
    else:
        extend = "both"
    _ = sp.draw_colorbar(label=f"{metric_bundle.metric.name} {metric_bundle.info_label}", pad=0.12, shrink=0.5, extend=extend, location="bottom", orientation='horizontal')

    if proj == 'laea':
        pass
    else:
        sp.ax.set_xlabel("R.A.", fontsize=12, labelpad=5)
        sp.ax.set_ylabel("Dec.", fontsize=12, labelpad=10)
    
    if title is not None:
        plt.title(title, fontsize='x-large', pad=30)
    return fig

In [ ]:
# # just for plotting purposes later (background)
from rubin_scheduler.scheduler.utils import get_current_footprint
fp, labels = get_current_footprint(nside=nside)

pp = tsurvey_info["skymap"]["map"]
alpha = np.where(pp >= 0, 1, fp['r'])
alpha = np.where(alpha > 1, 1, alpha)
bg_fp = np.where(fp['r'] == 0, np.nan, fp['r'])
bg_fp = np.where(bg_fp > 1, 1, bg_fp)

# Hack the footprint to only contain area only in i, z and
# in the region 300-324, -26 to -10
tmask = np.where(
    (tsurvey_info["skymap"]["ra"] > 300)
    & (tsurvey_info["skymap"]["ra"] < 324)
    & (tsurvey_info["skymap"]["dec"] > -26)
    & (tsurvey_info["skymap"]["dec"] < -10),
    1,
    0,
)

sv_fp = np.where(tsurvey_info['fp_array']['r'] * tmask > 0, 1, np.nan)
sv_fp = np.where(tsurvey_info['extra_templates_array']['i'] > 0, 0.5, sv_fp)

fig, ax = plt.subplots(figsize=(8, 7))

sp = skyproj.skyproj.McBrydeSkyproj(
    ax=ax,
    celestial=True,
    galactic=False,
    gridlines=True,
    n_grid_lon=8,
    n_grid_lat=5,
    #lat_0=-90,
    #lon_0=0,
    #extent=[0.0, 360.0, -90, 85],
)

sp.ax.set_xlabel("R.A.", fontsize=12, labelpad=9)
sp.ax.set_ylabel("Dec.", fontsize=12, labelpad=12)

mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
    bg_fp, cmap='grey_r', vmin=0, vmax=2, nest=False, zoom=False, zorder=0
)

mesh, lon_raster, lat_raster, values_raster = sp.draw_hpxmap(
    sv_fp, nest=False, zoom=False, zorder=1.5
)
fig.savefig("SV_v3_footprint.png", bbox_inches='tight')

In [ ]:
mask = np.where(sv_fp == 1, 1, np.nan)
len(np.where(sv_fp == 1)[0]) * hp.nside2pixarea(64, degrees=True)

In [ ]:
# SV survey initial simulation visits
opsdb = 'sv_sim_1.0.db'
sv_orig_name = opsdb.replace('.db', '')
conn = sqlite3.connect(opsdb)
sv_orig_visits = pd.read_sql("select * from observations", conn)

# baseline = 'baseline_v5.0.0_10yrs.db'
# baseline_name = baseline.replace('.db', '')
# conn = sqlite3.connect(os.path.join("/Users/lynnej/opsim/fbs_5.0", baseline))
# baseline_visits = pd.read_sql("select * from observations", conn)

In [ ]:
len(sv_orig_visits), #len(baseline_visits)

In [ ]:
baseline_nvisits = {}
baseline_coadd = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
    constraint = f"{b}"
    if b == 'all':
        opsvis = baseline_visits.to_records()
    else:
        opsvis = baseline_visits.query("band == @b").to_records()
    baseline_nvisits[b] = maf.MetricBundle(m_nvis, s, constraint, run_name=baseline_name)
    baseline_coadd[b] = maf.MetricBundle(m_coadd, s, constraint, run_name=baseline_name)
    g = maf.MetricBundleGroup({f'nvisits {b}': baseline_nvisits[b], f'coadd {b}': baseline_coadd[b]}, None)
    g.run_current(constraint, opsvis)

In [ ]:
vmax = 1100
fig = make_sv_plot(baseline_nvisits['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title=baseline_name)
fig.savefig(f"{baseline_name}_nvisits_skyproj.png", bbox_inches='tight')

In [ ]:
sv_orig_nvisits = {}
sv_orig_coadd = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
    constraint = f"{b}"
    if b == 'all':
        opsvis = sv_orig_visits.to_records()
    else:
        opsvis = sv_orig_visits.query("band == @b").to_records()
    sv_orig_nvisits[b] = maf.MetricBundle(m_nvis, s, constraint, run_name=sv_orig_name)
    sv_orig_coadd[b] = maf.MetricBundle(m_coadd, s, constraint, run_name=sv_orig_name)
    g = maf.MetricBundleGroup({f'nvisits {b}': sv_orig_nvisits[b], f'coadd {b}': sv_orig_coadd[b]}, None)
    g.run_current(constraint, opsvis)

In [ ]:
sv_orig_summary = pd.DataFrame([[np.nanmedian(sv_orig_nvisits[b].metric_values.filled(0) * mask) for b in sv_orig_nvisits],
                                  [np.nanmedian(sv_orig_coadd[b].metric_values.filled(0) * mask) for b in sv_orig_coadd]],
                                 columns=list(sv_orig_nvisits.keys()), index=['Nvisits', 'CoaddM5'])
display(sv_orig_summary.round(2))


vmax = np.percentile(sv_orig_nvisits['all'].metric_values.compressed(), 95)
fig = make_sv_plot(sv_orig_nvisits['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title="SV sim 1.0")
fig.savefig("sv_sim_1.0_nvisits.png", bbox_inches='tight')

In [ ]:
sv_simdays = []
for f in os.listdir('.'):
    if f.startswith('sv_') and os.path.isdir(f) and (f.endswith("20250620") or f.endswith("20250922")):
        for ff in os.listdir(f):
            if ff == f + ".db":
                sv_simdays.append(int(f.replace("sv_", "")))
sv_simdays = (np.sort(np.array(sv_simdays)))
sv_simdays_mjd = []
sv_simdays_datetime = []
for dayobs in sv_simdays:
    sv_simdays_mjd.append(rn_dayobs.day_obs_to_time(dayobs).mjd)
    sv_simdays_datetime.append(rn_dayobs.day_obs_to_time(dayobs).to_datetime())
sv_simdays, #sv_simdays_datetime

In [ ]:
sim_visits_dayobs = {}
for dayobs in sv_simdays:
    run_name = f"sv_{dayobs}"
    opsdb = os.path.join(run_name, f"sv_{dayobs}.db")
    conn = sqlite3.connect(opsdb)
    sim_visits_dayobs[dayobs] = pd.read_sql("select * from observations", conn)

In [ ]:
for dayobs, mjd in zip(sv_simdays, sv_simdays_mjd):
    q = sim_visits_dayobs[dayobs].query("observationStartMJD < @mjd")
    print(dayobs, mjd, len(q))

In [ ]:
# nvisits = {}
# coadd = {}
# m_nvis = maf.CountMetric(col='obs_start_mjd', metric_name="Nvisits")
# m_coadd = maf.Coaddm5Metric(m5_col='cat_m5')
# s = maf.HealpixSlicer(nside=64, lat_col='s_dec', lon_col='s_ra', rot_sky_pos_col_name='sky_rotation')
# for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
#     constraint = f"{b}"
#     if b == 'all':
#         opsvis = visits.to_records()
#     else:
#         opsvis = visits.query("band == @b").to_records()
#     print(constraint, len(opsvis))
#     nvisits[b] = maf.MetricBundle(m_nvis, s, constraint)
#     coadd[b] = maf.MetricBundle(m_coadd, s, constraint)
#     g = maf.MetricBundleGroup({f'nvisits {b}': nvisits[b], f'coadd {b}': coadd[b]}, None)
#     g.run_current(constraint, opsvis)

In [ ]:
# End of survey values
sim_nvisits = {}
sim_coadd = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for dayobs in sv_simdays:
    sim_nvisits[dayobs] = {}
    sim_coadd[dayobs] = {}
    for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
        constraint = f"{b}"
        if b == 'all':
            opsvis = sim_visits_dayobs[dayobs].to_records()
        else:
            opsvis = sim_visits_dayobs[dayobs].query("band == @b").to_records()
        sim_nvisits[dayobs][b] = maf.MetricBundle(m_nvis, s, constraint)
        sim_coadd[dayobs][b] = maf.MetricBundle(m_coadd, s, constraint)
        g = maf.MetricBundleGroup({f'nvisits {b}': sim_nvisits[dayobs][b], f'coadd {b}': sim_coadd[dayobs][b]}, None)
        g.run_current(constraint, opsvis)

In [ ]:
sim_summary = {}
for dayobs in sv_simdays:
    sim_summary[dayobs] = pd.DataFrame([[np.nanmedian(sim_nvisits[dayobs][b].metric_values.filled(0) * mask) for b in sim_nvisits[dayobs]],
                      [np.nanmedian(sim_coadd[dayobs][b].metric_values.filled(0) * mask) for b in sim_coadd[dayobs]]],
                      columns=list(sim_nvisits[dayobs].keys()), index=[f'Nvisits', f'CoaddM5'])
sim_summary = pd.concat(sim_summary)
display(sv_orig_summary.round(2))
display(sim_summary.round(2)) 

In [ ]:
for dayobs in sim_nvisits:
    nvisits = sim_nvisits[dayobs]
    out_dir = f"sv_{dayobs}"
    fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(16, 10),)
    axdict = {"u": ax[0][0], "g": ax[0][1], "r": ax[0][2],
              "i": ax[1][0], "z": ax[1][1], "y": ax[1][2], "all": None}
    for b in ["u", "g", "r", "i", "z", "y"]:
        if len(nvisits[b].metric_values.compressed()) > 1:
            vmax = np.percentile(nvisits[b].metric_values.compressed(), 95)
        else:
            vmax = None
        fig = make_sv_plot(nvisits[b], proj='McBryde', vmax=vmax, ax=axdict[b], title=f"SV {dayobs} band {b}")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"sv_from_{dayobs}_nvisits_band.png"), bbox_inches='tight')
    
    vmax = np.percentile(nvisits['all'].metric_values.compressed(), 95)
    fig = make_sv_plot(nvisits['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title=f"SV from {dayobs}")
    fig.savefig(os.path.join(out_dir, f"sv_from_{dayobs}_nvisits.png"), bbox_inches='tight')
    plt.close()

In [ ]:
#sim_summary.to_hdf("sim_summary_20250729.h5", key='end_of_survey')

In [ ]:
# plt.figure(figsize=(8, 6))
# vals = []
# nvis_idxs = sim_summary.index.to_flat_index()[0::2]
# coadd_idxs = sim_summary.index.to_flat_index()[1::2]
# plt.plot(sv_simdays_datetime, sim_summary.loc[coadd_idxs, "r"], color='orange', marker='o', linestyle='-')
# plt.axhline(sv_orig_summary.loc['CoaddM5', "r"], color='teal', linestyle='-', label=sv_orig_name)
# plt.xticks(rotation=45)
# plt.xlabel("Date", fontsize='x-large')
# plt.ylabel("End of Survey CoaddM5 r band", fontsize='x-large')
# plt.title("SV survey", fontsize='x-large')
# plt.savefig("../figures/sv_coadd_prediction_20250729.png", bbox_inches='tight')

In [ ]:
# plt.figure(figsize=(8, 6))
# vals = []
# nvis_idxs = sim_summary.index.to_flat_index()[0::2]
# coadd_idxs = sim_summary.index.to_flat_index()[1::2]
# plt.plot(sv_simdays_datetime, sim_summary.loc[nvis_idxs, "all"], color='orange', marker='o', linestyle='-')
# plt.axhline(sv_orig_summary.loc['Nvisits', "all"], color='teal', linestyle='-', label=sv_orig_name)
# plt.xticks(rotation=45)
# plt.xlabel("Date", fontsize='x-large')
# plt.ylabel("End of Survey Number of Visits per pointing", fontsize='x-large')
# plt.title("SV survey", fontsize='x-large')
# plt.savefig("../figures/sv_nvis_prediction_20250729.png", bbox_inches='tight')

In [ ]:
# current place in survey values
cur_sim_nvisits = {}
cur_sim_coadd = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for dayobs, mjd in zip(sv_simdays, sv_simdays_mjd):
    q = sim_visits_dayobs[dayobs].query("observationStartMJD < @mjd")
    if len(q) == 0:
        continue
    cur_sim_nvisits[dayobs] = {}
    cur_sim_coadd[dayobs] = {}
    for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
        constraint = f"{b} dayobs"
        if b == 'all':
            opsvis = q.to_records()
        else:
            opsvis = q.query("band == @b").to_records()
        cur_sim_nvisits[dayobs][b] = maf.MetricBundle(m_nvis, s, constraint)
        cur_sim_coadd[dayobs][b] = maf.MetricBundle(m_coadd, s, constraint)
        g = maf.MetricBundleGroup({f'nvisits {b}': cur_sim_nvisits[dayobs][b], f'coadd {b}': cur_sim_coadd[dayobs][b]}, None)
        # If no data, make a metric value entirely masked
        if len(opsvis) == 0:
            cur_sim_nvisits[dayobs][b].metric_values = np.ma.MaskedArray(np.zeros(len(s)), np.ones(len(s)))
            cur_sim_coadd[dayobs][b].metric_values = np.ma.MaskedArray(np.zeros(len(s)), np.ones(len(s)))
        else:
            g.run_current(constraint, opsvis)

In [ ]:
cur_sim_summary = {}
for dayobs in cur_sim_nvisits:
    cur_sim_summary[dayobs] = pd.DataFrame([[np.nanmedian(cur_sim_nvisits[dayobs][b].metric_values.filled(0) * mask) for b in cur_sim_nvisits[dayobs]],
                      [np.nanmedian(cur_sim_coadd[dayobs][b].metric_values.filled(0) * mask) for b in cur_sim_coadd[dayobs]]],
                      columns=list(cur_sim_nvisits[dayobs].keys()), index=[f'Nvisits', f'CoaddM5'])
cur_sim_summary = pd.concat(cur_sim_summary)
display(cur_sim_summary.round(2))

In [ ]:
rst_table = tabulate(pd.DataFrame(cur_sim_summary.round(1)), headers='keys', tablefmt='rst')
print(rst_table)

In [ ]:
#cur_sim_summary.to_hdf("sim_summary_20250729.h5", key='current_survey')

In [ ]:
for dayobs in cur_sim_nvisits:
    nvisits = cur_sim_nvisits[dayobs]
    out_dir = f"sv_{dayobs}"
    fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(16, 10),)
    axdict = {"u": ax[0][0], "g": ax[0][1], "r": ax[0][2],
              "i": ax[1][0], "z": ax[1][1], "y": ax[1][2], "all": None}
    for b in ["u", "g", "r", "i", "z", "y"]:
        if len(nvisits[b].metric_values.compressed()) > 1:
            vmax = np.percentile(nvisits[b].metric_values.compressed(), 95)
        else:
            vmax = None
        fig = make_sv_plot(nvisits[b], proj='McBryde', vmax=vmax, ax=axdict[b], title=f"SV @ {dayobs} band {b}")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, f"sv_@_{dayobs}_nvisits_band.png"), bbox_inches='tight')
    
    vmax = np.percentile(nvisits['all'].metric_values.compressed(), 95)
    fig = make_sv_plot(nvisits['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title=f"SV @ {dayobs}")
    fig.savefig(os.path.join(out_dir, f"sv_@_{dayobs}_nvisits.png"), bbox_inches='tight')
    plt.close()

In [ ]:
for dayobs, mjd in zip(sv_simdays, sv_simdays_mjd):
    sim_visits = sim_visits_dayobs[dayobs]
    q = sim_visits_dayobs[dayobs].query("observationStartMJD < @mjd")
    if len(q) == 0:
        continue
    
    out_dir = f"sv_{dayobs}"

    plt.figure(figsize=(8, 6))

    nights = np.arange(0, sv_orig_visits.night.max())
    _ = plt.hist(sv_orig_visits.night, bins=nights, histtype='step', cumulative=True, label='sv_sim_1.0', linewidth=2)
    
    #gsim_visits = sim_visits.query("seeingFwhmEff < 1.2 and night >= 13")
    #_ = plt.hist(gsim_visits.night, bins=nights, histtype='step', cumulative=True, label='GQ_sim_1.0', linewidth=1)
    
    _ = plt.hist(sim_visits.night, bins=nights, histtype='step', cumulative=True, label=f'sv_{dayobs}', linewidth=2)

    #gsim_visits = new_sim_visits.query("seeingFwhmEff < 1.2 and night >= 13")
    #_ = plt.hist(gsim_visits.night, bins=nights, histtype='step', cumulative=True, label=f'GQ_sv_{dayobs}', linewidth=1)
    
    nights = np.arange(0, q.night.max() + 1)
    _ = plt.hist(q.night, bins=nights, histtype='step', cumulative=True, label=f'onsky {dayobs}', linewidth=2)
    
    #good_visits = visits.query("fwhm_eff < 1.2 and day_obs >= 20250703")
    #_ = plt.hist(good_visits.night, bins=nights, histtype='step', cumulative=True, label=f'GQ sv_{dayobs}', linewidth=2)
    
    
    plt.axvline(q.night.max(), color='black', linewidth=2)
    
    #plt.axvline((Time("2025-08-22T12:00:00", scale='tai') - survey_info['survey_start']).jd, color='black', linestyle=':')
    
    plt.xlim(0, sim_visits.night.max() - 1)
    plt.grid(alpha=0.5)
    plt.xlabel("Night of SV survey")
    plt.ylabel("Cumulative number of visits")
    plt.legend()
    plt.savefig(os.path.join(out_dir, f'cumulative_visits_{dayobs}.png'))
    plt.close()

In [ ]:
for dayobs, mjd in zip(sv_simdays[-1:], sv_simdays_mjd[-1:]):
    sim_visits = sim_visits_dayobs[dayobs]
    q = sim_visits_dayobs[dayobs].query("observationStartMJD < @mjd")
    if len(q) == 0:
        continue
    
    out_dir = f"sv_{dayobs}"

    plt.figure(figsize=(8, 6))

    nights = np.arange(0, sv_orig_visits.night.max())
    _ = plt.hist(sv_orig_visits.night, bins=nights, histtype='step', cumulative=True, label='sv_sim_1.0', linewidth=2)
    
    #gsim_visits = sim_visits.query("seeingFwhmEff < 1.2 and night >= 13")
    #_ = plt.hist(gsim_visits.night, bins=nights, histtype='step', cumulative=True, label='GQ_sim_1.0', linewidth=1)
    
    _ = plt.hist(sim_visits.night, bins=nights, histtype='step', cumulative=True, label=f'sv_{dayobs}', linewidth=2)

    #gsim_visits = new_sim_visits.query("seeingFwhmEff < 1.2 and night >= 13")
    #_ = plt.hist(gsim_visits.night, bins=nights, histtype='step', cumulative=True, label=f'GQ_sv_{dayobs}', linewidth=1)
    
    nights = np.arange(0, q.night.max() + 1)
    _ = plt.hist(q.night, bins=nights, histtype='step', cumulative=True, label=f'onsky {dayobs}', linewidth=2)
    
    #good_visits = visits.query("fwhm_eff < 1.2 and day_obs >= 20250703")
    #_ = plt.hist(good_visits.night, bins=nights, histtype='step', cumulative=True, label=f'GQ sv_{dayobs}', linewidth=2)
    
    
    plt.axvline(q.night.max(), color='black', linewidth=2)
    
    #plt.axvline((Time("2025-08-22T12:00:00", scale='tai') - survey_info['survey_start']).jd, color='black', linestyle=':')
    
    plt.xlim(0, sim_visits.night.max() - 1)
    plt.grid(alpha=0.5)
    plt.xlabel("Night of SV survey")
    plt.ylabel("Cumulative number of visits")
    plt.legend()
    #plt.savefig(os.path.join(out_dir, f'cumulative_visits_{dayobs}.png'))
    #plt.close()

In [ ]:
from tabulate import tabulate
import datetime

def write_template(dayobs, nvisits_total, nvis_summary):
    base_url = "https://s3df.slac.stanford.edu/data/rubin/sim-data/sv/sv_progress_databases"
    with open(f"../progress/sv_status/sv_{dayobs}.rst", "w") as rst:
        rst.write(f".. _SV_{dayobs}:\n")
        rst.write("\n \n")
        rst.write("####################\n")
        rst.write(f"SV {dayobs}\n")
        rst.write("####################\n")
        rst.write("\n \n")
        rst.write("Comments here\n")
        rst.write("\n")
        rst.write("Predictions for end of SV\n")
        rst.write("=========================\n")
        rst.write("\n")
        rst.write("The current predictions from now to the end of the SV survey still contain significant uncertainty, ")
        rst.write("primarily due to uncertainty about time which must be spent in other engineering activities, but also ")
        rst.write("keeping in mind that over short time periods, the impact of weather is much more significant than it is ")
        rst.write("over the period of years. ")
        rst.write("\n")
        rst.write("\n")
        rst.write(f".. figure:: ../../figures/sv_{dayobs}/sv_from_{dayobs}_nvisits.png\n")
        rst.write("  :width: 650\n")
        rst.write("  :alt: SV survey acquired and simulated visits, all bands\n")
        rst.write("\n")
        rst.write(f".. figure:: ../../figures/sv_{dayobs}/sv_from_{dayobs}_nvisits_band.png\n")
        rst.write("  :width: 650\n")
        rst.write("  :alt: SV survey acquired and simulated visits, per band.\n")
        rst.write("\n")
        rst.write(f"Acquired (20250620 to {dayobs}) plus simulated ({dayobs} to 20250922) visits in SV. \n")
        rst.write("\n")
        rst.write("Beyond downtime due to weather, the plot below illustrates lost to expected engineering activities ")
        rst.write("(black lines = downtime). Downtime during visits that have already been acquired is based on the actual time ")
        rst.write("that was used for the SV survey. Downtime in the future (beyond the vertical red dashed line), is estimated. ")
        rst.write("Estimates include:  time spent on other commissioning activities to improve image quality, closing the dome ")
        rst.write("two hours prior to 0-degree sunrise, and blocking the first thirty minutes or so of the night for start of night activities.\n")
        rst.write("\n")
        rst.write(f".. figure:: ../../figures/sv_{dayobs}/onsky_downtime.png\n")
        rst.write("  :width: 500\n")
        rst.write("  :alt: On-sky time availability.\n")
        rst.write("Downtime from the past (based on actual SV time onsky) and into the future (in the simulation). \n")
        rst.write("\n")
        rst.write("The cumulative visits so far, extended through the end of SV with a simulation ")
        rst.write("folding in the above downtime are shown in the next figure. The original ")
        rst.write("sim_sv_1.0' (with no weather downtime) and 'sv_20250620' (like sim_sv_1.0 but with weather downtime) ")
        rst.write("are also shown for comparison.")
        rst.write("\n")
        rst.write("\n")
        rst.write(f".. figure:: ../../figures/sv_{dayobs}/cumulative_visits_{dayobs}.png\n")
        rst.write("  :width: 600\n")
        rst.write("  :alt: Cumulative number of visits in the SV survey, to date and to end of SV.\n")
        rst.write("Cumulative number of visits in the SV survey, to date and to end of SV. \n")
        rst.write("\n")
        rst.write(f"A simulation database containing acquired visits up to {dayobs} and extended ")
        rst.write(f"to the end of the SV survey: `sv_{dayobs}.db <{base_url}/sv_{dayobs}/sv_{dayobs}.db>`_. \n")
        rst.write("Note that these databases contain preliminary visit metadata; not all visits will ")
        rst.write("successfully pass through processing into data releases, and metadata may change with ")
        rst.write("further processing or information. The system is still under commissioning. \n")
        rst.write("\n")
        rst.write("Visits acquired in the SV survey to date\n")
        rst.write("========================================\n")
        rst.write("\n")
        rst.write(f"As of {dayobs}, the SV survey has acquired a total of {nvisits_total} visits, ")
        rst.write("excluding known bad visits, and still including visits with a wide range of ")
        rst.write("data quality, due to both cloud extinction and delivered IQ. \n")
        rst.write("\n")
        rst.write(f"The median numbers of visits, coadded depth, and effective exposure time per pointing within the primary wide SV survey area to date are: \n")
        rst.write("\n")
        rst_table = tabulate(pd.DataFrame(summary), headers='keys', tablefmt='rst')
        rst.write(rst_table)
        rst.write("\n")
        rst.write("\n")
        rst.write(f".. figure:: ../../figures/sv_{dayobs}/sv_@_{dayobs}_nvisits.png\n")
        rst.write("  :width: 650\n")
        rst.write("  :alt: SV survey acquired visits, all bands.\n")
        rst.write(f".. figure:: ../../figures/sv_{dayobs}/sv_@_{dayobs}_nvisits_band.png\n")
        rst.write("  :width: 650\n")
        rst.write("  :alt: SV survey acquired visits, per band.\n")
        rst.write(f"Acquired to date (20250620 to {dayobs}) visits in SV.\n")
        rst.write("\n")
        rst.write("\n")
        rst.write(" \n \n \n")
        rst.write(".. toctree::\n")
        rst.write("    :maxdepth: 2\n")
        rst.write("    :titlesonly:\n")
        rst.write("    :glob: \n")
        rst.write("\n")
        rst.write(".. admonition:: Last Updated\n")
        rst.write(" \n")
        rst.write(f"  Last Updated {datetime.datetime.now().strftime("%Y/%m/%d")} \n")
        rst.write("..   * \n")

# for dayobs, mjd in zip(sv_simdays[1:], sv_simdays_mjd[1:]):
#     sim_visits = sim_visits_dayobs[dayobs]
#     q = sim_visits_dayobs[dayobs].query("observationStartMJD < @mjd")
#     summary =  cur_sim_summary.loc[(dayobs, 'Nvisits')].astype(int)
#     write_template(dayobs, len(q), summary)


for dayobs, mjd in zip(tt[-1:], tt_mjd[-1:]):
    sim_visits = sim_visits_dayobs_with_clouds[dayobs]
    q = sim_visits_dayobs_with_clouds[dayobs].query("observationStartMJD < @mjd")
    summary = cur_sim_summary.round(1)
    #summary = cloud_sim_summary.loc[(dayobs, 'Nvisits')].astype(int)
    write_template(dayobs, len(q), summary)

In [ ]:
sim_visits_dayobs_with_clouds = {}
tt = np.array([20250620, 20250920], int)
tt_mjd = np.array([rn_dayobs.day_obs_to_time(dayobs).mjd for dayobs in tt])
for dayobs in tt:
    run_name = f"sv_{dayobs}"
    opsdb = os.path.join(run_name, f"sv_{dayobs}.db")
    conn = sqlite3.connect(opsdb)
    sim_visits_dayobs_with_clouds[dayobs] = pd.read_sql("select * from observations", conn)

In [ ]:
dayobs = 20250920
out_dir = f"sv_{dayobs}"

plt.figure(figsize=(8, 6))

nights = np.arange(0, sv_orig_visits.night.max())
_ = plt.hist(sv_orig_visits.night, bins=nights, histtype='step', cumulative=True, label='sv_sim_1.0', linewidth=2)


for dayobs, mjd in zip(tt, tt_mjd):

    #sim_visits = sim_visits_dayobs[dayobs]
    #_ = plt.hist(sim_visits.night, bins=nights, histtype='step', cumulative=True, label=f'sv_{dayobs}', linewidth=1, linestyle='-.')

    sim_visits = sim_visits_dayobs_with_clouds[dayobs]
    _ = plt.hist(sim_visits.night, bins=nights, histtype='step', cumulative=True, label=f'sv_{dayobs}', linewidth=1)

q = sim_visits_dayobs_with_clouds[dayobs].query("observationStartMJD < @mjd")
if len(q) > 0:
    #night_max = q.night.max()
    night_max = int(Time("2025-09-03T12:00:00").mjd - Time("2025-06-20T12:00:00").mjd)
    nights = np.arange(0, night_max + 1)
    _ = plt.hist(q.night, bins=nights, histtype='step', cumulative=True, label=f'onsky {dayobs}', linewidth=2)
    
plt.axvline(night_max, color='black', linewidth=2)


plt.xlim(0, sim_visits.night.max() - 1)
plt.grid(alpha=0.5)
plt.xlabel("Night of SV survey", fontsize='large')
plt.ylabel("Cumulative number of visits", fontsize='large')
plt.legend()
plt.savefig(os.path.join(out_dir, f'cumulative_visits_{dayobs}.png'))
#plt.close()

In [ ]:
len(dict())

In [ ]:
# End of survey values
cloud_sim_nvisits = {}
cloud_sim_coadd = {}
m_nvis = maf.CountMetric(col='observationStartMJD', metric_name = "Nvisits")
m_coadd = maf.Coaddm5Metric(m5_col='fiveSigmaDepth')
s = maf.HealpixSlicer(nside=64)
for dayobs in tt[-1:]:
    cloud_sim_nvisits[dayobs] = {}
    cloud_sim_coadd[dayobs] = {}
    for b in ['u', 'g', 'r', 'i', 'z', 'y', 'all']:
        constraint = f"{b}"
        if b == 'all':
            opsvis = sim_visits_dayobs_with_clouds[dayobs].to_records()
        else:
            opsvis = sim_visits_dayobs_with_clouds[dayobs].query("band == @b").to_records()
        cloud_sim_nvisits[dayobs][b] = maf.MetricBundle(m_nvis, s, constraint)
        cloud_sim_coadd[dayobs][b] = maf.MetricBundle(m_coadd, s, constraint)
        g = maf.MetricBundleGroup({f'nvisits {b}': cloud_sim_nvisits[dayobs][b], f'coadd {b}': cloud_sim_coadd[dayobs][b]}, None)
        g.run_current(constraint, opsvis)

In [ ]:
cloud_sim_summary = {}
for dayobs in cloud_sim_nvisits:
    cloud_sim_summary[dayobs] = pd.DataFrame([[np.nanmedian(cloud_sim_nvisits[dayobs][b].metric_values.filled(0) * mask) for b in cloud_sim_nvisits[dayobs]],
                      [np.nanmedian(cloud_sim_coadd[dayobs][b].metric_values.filled(0) * mask) for b in cloud_sim_coadd[dayobs]]],
                      columns=list(cloud_sim_nvisits[dayobs].keys()), index=[f'Nvisits', f'CoaddM5'])
cloud_sim_summary = pd.concat(cloud_sim_summary)
display(cloud_sim_summary.round(2))

dayobs = list(cloud_sim_nvisits.keys())[-1]
vmax = np.percentile(cloud_sim_nvisits[dayobs]['all'].metric_values.compressed(), 95)
fig = make_sv_plot(cloud_sim_nvisits[dayobs]['all'], proj='mcbryde', vmin=None, vmax=vmax, ax=None, add_bg=True, title=f"sv_{dayobs}")

In [ ]:
mjd
v = sim_visits_dayobs_with_clouds[np.int64(20250903)].query("observationStartMJD < @mjd")
print(v.slewTime.mean(), v.slewTime.median())
v = sim_visits_dayobs_with_clouds[np.int64(20250903)].query("observationStartMJD < @mjd and slewTime < 150")
print(v.slewTime.mean(), v.slewTime.median())
v = sim_visits_dayobs_with_clouds[np.int64(20250903)].query("observationStartMJD < @mjd and slewTime >= 150 and slewTime < 36000")
print(v.slewTime.sum()/60/60/24)

In [ ]:
bins = np.arange(0.5, 2.6, 0.001)
for b in 'ugrizy':
    q = visits.query("band == @b")
    _ = plt.hist(q.fwhm_eff, bins=bins, cumulative=True, density=True, color=band_colors[b], histtype='step', linestyle='-.', label=f"SV {b}")
    print('sv', b, f"{q.fwhm_eff.median():.2f}")
    q = sim_visits.query("band == @b")
    _ = plt.hist(q.seeingFwhmEff, bins=bins, cumulative=True, density=True, color=band_colors[b], histtype='step', linestyle='-', label=f"Sim {b}")
    print('sim', b, f"{q.seeingFwhmEff.median():.2f}")

plt.axhline(.5, color='gray', alpha=0.7)
plt.legend(loc=(1.01, 0.1))
plt.grid(alpha=0.4)
plt.xlim(0.4, 2.0)
plt.xlabel("FWHM (arcsec)")

In [ ]:
len(good_visits)/len(sim_visits), len(visits) / len(sim_visits), (Time("2025-07-20T12:00:00") - survey_info['survey_start']).jd / (survey_info['survey_end'] - survey_info['survey_start']).jd

In [ ]:
# eric's progress reports
# example in schedview reports - https://s3df.slac.stanford.edu/data/rubin/sim-data/schedview/sample_reports/progress_mockup.html
# example notebook https://usdf-rsp-int.slac.stanford.edu/schedview-static-pages/progress/sv/devel_sample.html
# example rendered https://s3df.slac.stanford.edu/data/rubin/sim-data/schedview/reports/svprogress/lsstcam/2025/07/20/svprogress_2025-07-20.html

In [ ]:
from sv_survey import 
import fbs_config_sv_survey as fbs_config
_ = importlib.reload(fbs_config)

nside, starting_scheduler = fbs_config.get_scheduler()
scheduler = copy.deepcopy(starting_scheduler)

In [ ]:
skymap = survey_info['skymap']
mm = np.where((skymap['dec'] > -45) & (skymap['dec'] < -15) & (skymap['ra'] > 30) & (skymap['ra'] < 110))[0]
len(mm) * hp.nside2pixarea(survey_info['nside'], degrees=True)

In [ ]:
Time("2025-12-21T12:00:00", format="isot", scale="utc") -  Time("2025-10-22T12:00:00", format="isot", scale="utc")